# Week 8: Distributed Machine Learning

This notebook accompanies the Week 8 lecture on **Distributed Machine Learning** and uses the **POGOH** Pittsburgh bike-share dataset as a running example throughout.

It follows the lecture roadmap:

**Foundations**
1. Why ML becomes a systems problem
2. Distributed feature engineering
3. Forms of parallelism in ML
4. Distributed optimization & trade-offs

**Systems & Application**
5. Spark MLlib abstractions & distributed pipelines
6. Which models scale well and why
7. Evaluation & leakage at scale
8. End-to-end operational workflow

## Learning goals

By the end of this notebook, you should be able to:

1. Explain the three constraints (memory, compute, network) that make ML a systems problem.
2. Engineer distributed features using Spark DataFrame operations and identify data skew.
3. Explain what data-parallel training looks like inside Spark MLlib.
4. Build a full Spark MLlib `Pipeline` (StringIndexer, OHE, VectorAssembler, classifier).
5. Implement a **time-based train/test split** and explain why it prevents leakage.
6. Compare distributed scaling characteristics of Logistic Regression vs Random Forest.
7. Apply K-Means clustering to station profiles and evaluate with silhouette score.

## Prediction task

> **Will this POGOH trip last 20 minutes (1,200 s) or longer?**

We use only **pre-trip features**: start station, rider type, hour, day of week, month.

## Leakage warning

These columns are **off-limits** — they are only known after the trip ends:
- `Duration` (the raw label source)
- `End Station Id` / `End Station Name`
- `End Date`

---
## Setup

In [ ]:
# Uncomment if running outside the course Docker environment
# !pip install pyspark

In [1]:
import time

from pyspark.sql import SparkSession, functions as F, types as T
from pyspark.ml import Pipeline
from pyspark.ml.feature import (
    StringIndexer, OneHotEncoder, VectorAssembler, StandardScaler
)
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import (
    BinaryClassificationEvaluator,
    MulticlassClassificationEvaluator,
    ClusteringEvaluator,
)

spark = (
    SparkSession.builder
    .appName("Week08_DistributedML")
    # shuffle.partitions controls post-shuffle partition count.
    # Default (200) is too high for small datasets; 50 is reasonable for the sample.
    .config("spark.sql.shuffle.partitions", "50")
    .getOrCreate()
)

spark

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/03/31 19:47:25 WARN Utils: Your hostname, MacBook-Pro.local, resolves to a loopback address: 127.0.0.1; using 172.31.33.116 instead (on interface en0)
26/03/31 19:47:25 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/31 19:47:26 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


---
## Part 1 — Why ML Becomes a Systems Problem

At small scale, ML is about algorithm selection and parameter tuning.  
At large scale it becomes a **data, systems, and communication** problem.

| Constraint | What breaks |
|---|---|
| **Memory** | Dataset no longer fits in one machine's RAM |
| **Compute** | Matrix ops over billions of rows are prohibitively slow on one core |
| **Network** | Distributed speedup is capped by synchronisation cost and data movement |

Even the POGOH sample illustrates all three:
- Feature aggregations trigger **shuffles** across the cluster.
- A few stations are dramatically more popular — this is **data skew**.
- Fitting logistic regression requires repeated gradient passes with cross-executor aggregation.

In [2]:
# Adjust path if needed. Use pogoh_full.csv for the full multi-year dataset.
DATA_PATH = "../data/pogoh_full.csv"

raw = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(DATA_PATH)
)

raw.printSchema()

root
 |-- Closed Status: string (nullable = true)
 |-- Duration: integer (nullable = true)
 |-- Start Station Id: integer (nullable = true)
 |-- Start Date: timestamp (nullable = true)
 |-- Start Station Name: string (nullable = true)
 |-- End Date: timestamp (nullable = true)
 |-- End Station Id: integer (nullable = true)
 |-- End Station Name: string (nullable = true)
 |-- Rider Type: string (nullable = true)
 |-- source_file: string (nullable = true)



In [3]:
raw.show(5, truncate=False)

+-------------+--------+----------------+-------------------+---------------------------+-------------------+--------------+--------------------------------------+----------+---------------+
|Closed Status|Duration|Start Station Id|Start Date         |Start Station Name         |End Date           |End Station Id|End Station Name                      |Rider Type|source_file    |
+-------------+--------+----------------+-------------------+---------------------------+-------------------+--------------+--------------------------------------+----------+---------------+
|NORMAL       |299     |51              |2024-04-30 23:53:00|Coltart Ave & Forbes Ave   |2024-04-30 23:58:00|20            |Boulevard of the Allies & Parkview Ave|MEMBER    |april-2024.xlsx|
|NORMAL       |399     |28              |2024-04-30 23:49:00|Fifth Ave & S Bouquet St   |2024-04-30 23:56:00|10            |Zulema St & Coltart Ave               |MEMBER    |april-2024.xlsx|
|NORMAL       |409     |34              |2024

In [4]:
# count() triggers a full data scan (expensive at scale).
# getNumPartitions() reads metadata only (cheap).
# Each partition maps to one task on one executor core.
print("Row count:  ", raw.count())
print("Partitions: ", raw.rdd.getNumPartitions())

# Rule of thumb for production: aim for 100-200 MB per partition.

Row count:   1034643
Partitions:  18


---
## Part 2 — Distributed Feature Engineering

Feature engineering is often **more expensive** than fitting the model itself.  
Every `groupBy` or multi-table `join` triggers a **network shuffle** — Spark
redistributes rows so all records sharing the same key land on the same executor.

We engineer the following from `Start Date`:

| Feature | Rationale |
|---|---|
| `month` | Seasonal demand |
| `day_of_week` | Weekday commute vs weekend recreation |
| `hour` | Rush hour vs leisure |
| `label` | 1 if `Duration >= 1200` s, else 0 |

And **training-only aggregates** (details in Part 3):

| Feature | Description |
|---|---|
| `station_trip_count` | Historical trip volume at start station |
| `station_long_trip_rate` | Historical fraction of long trips from start station |

In [5]:
trips = (
    raw
    # Standardise column names (no spaces — required for MLlib compatibility)
    .withColumnRenamed("Start Station Id",   "start_station_id")
    .withColumnRenamed("Start Station Name", "start_station_name")
    .withColumnRenamed("Rider Type",         "rider_type")
    .withColumnRenamed("Duration",           "duration_sec")
    .withColumnRenamed("Start Date",         "start_ts")
    # Parse timestamp
    .withColumn("start_ts", F.to_timestamp("start_ts", "yyyy-MM-dd HH:mm:ss"))
    # Temporal features
    .withColumn("month",       F.month("start_ts").cast("double"))
    .withColumn("day_of_week", F.dayofweek("start_ts").cast("double"))  # 1=Sun..7=Sat
    .withColumn("hour",        F.hour("start_ts").cast("double"))
    # Binary label
    .withColumn("label",       (F.col("duration_sec") >= 1200).cast("double"))
    # Drop rows with unparseable timestamps or null labels
    .filter(F.col("start_ts").isNotNull() & F.col("label").isNotNull())
    # Keep only columns we need — post-trip columns are intentionally excluded
    .select(
        "start_ts", "start_station_id", "start_station_name",
        "rider_type", "month", "day_of_week", "hour", "label"
    )
)

trips.show(5, truncate=False)
print(f"Rows after cleaning: {trips.count()}")

+-------------------+----------------+---------------------------+----------+-----+-----------+----+-----+
|start_ts           |start_station_id|start_station_name         |rider_type|month|day_of_week|hour|label|
+-------------------+----------------+---------------------------+----------+-----+-----------+----+-----+
|2024-04-30 23:53:00|51              |Coltart Ave & Forbes Ave   |MEMBER    |4.0  |3.0        |23.0|0.0  |
|2024-04-30 23:49:00|28              |Fifth Ave & S Bouquet St   |MEMBER    |4.0  |3.0        |23.0|0.0  |
|2024-04-30 23:46:00|34              |N Dithridge St & Centre Ave|MEMBER    |4.0  |3.0        |23.0|0.0  |
|2024-04-30 23:43:00|10              |Zulema St & Coltart Ave    |MEMBER    |4.0  |3.0        |23.0|0.0  |
|2024-04-30 23:42:00|13              |S Bouquet Ave & Sennott St |MEMBER    |4.0  |3.0        |23.0|0.0  |
+-------------------+----------------+---------------------------+----------+-----+-----------+----+-----+
only showing top 5 rows
Rows after cl

### Data Skew: Inspecting Hot Keys

**Data skew** occurs when some keys appear far more often than others.  
In a `groupBy`, the executor processing the hot-key partition does far more work
than its peers — it becomes a **straggler** that delays the entire synchronous job.

This is one of the most common performance problems in distributed feature engineering.

In [6]:
# Station popularity — look for hot keys
trips.groupBy("start_station_name").count().orderBy(F.desc("count")).show(15, truncate=False)

# Class balance
trips.groupBy("label").count().orderBy("label").show()

# Rider type breakdown
trips.groupBy("rider_type").count().orderBy(F.desc("count")).show()

print("Unique stations:", trips.select("start_station_id").distinct().count())

+----------------------------------------+------+
|start_station_name                      |count |
+----------------------------------------+------+
|S Bouquet Ave & Sennott St              |101130|
|Atwood St & Bates St                    |63467 |
|Boulevard of the Allies & Parkview Ave  |62336 |
|N Dithridge St & Centre Ave             |59587 |
|Zulema St & Coltart Ave                 |47333 |
|Coltart Ave & Forbes Ave                |46918 |
|Fifth Ave & S Bouquet St                |46732 |
|O'Hara St & University Place            |33052 |
|Allequippa St & Darragh St              |32337 |
|Schenley Dr & Schenley Dr Ext           |31364 |
|Forbes Ave at TCS Hall (CMU Campus)     |28438 |
|Forbes Ave & Schenley Dr                |28234 |
|O'Hara St and University Place          |24934 |
|S 27th St & Sidney St. (Southside Works)|23828 |
|Ivy St & Walnut St                      |20042 |
+----------------------------------------+------+
only showing top 15 rows
+-----+------+
|label| co

In [7]:
# Long-trip rate by hour of day — captures the commute vs leisure signal
(
    trips.groupBy("hour")
    .agg(F.count("*").alias("n"), F.avg("label").alias("long_trip_rate"))
    .orderBy("hour")
).show(24)

+----+-----+-------------------+
|hour|    n|     long_trip_rate|
+----+-----+-------------------+
| 0.0|17109|0.17978841545385468|
| 1.0|12646|0.18140123359164953|
| 2.0| 7834|0.19249425580801635|
| 3.0| 3838| 0.2037519541427827|
| 4.0| 2202|0.20072661217075385|
| 5.0| 3815|0.10878112712975098|
| 6.0|10585|0.08880491261218705|
| 7.0|27045|0.09432427435755222|
| 8.0|36906|0.08630033056955509|
| 9.0|44156|0.08854968747169127|
|10.0|48489|0.12128523995132917|
|11.0|55729|0.14495146153708124|
|12.0|71531|0.14128140246885965|
|13.0|68185| 0.1586272640610105|
|14.0|76406|0.15311624741512447|
|15.0|79708| 0.1583780799919707|
|16.0|86206|0.16535971974108532|
|17.0|89886| 0.1663106601695481|
|18.0|74679|0.17429263916228124|
|19.0|65863|0.17167453653796516|
|20.0|55617|0.15824298326051386|
|21.0|41433|0.16472377090724785|
|22.0|31067|0.17957961824443944|
|23.0|23708|0.18352454867555257|
+----+-----+-------------------+



---
## Part 3 — Time-Based Split and Leakage Prevention

### The wrong way: random split

A random split mixes all time periods in train and test.  
If we then compute station aggregates from the full dataset,
**test-period information leaks into training features** — the model sees the future.

Symptom: offline AUC looks great; production performance disappoints.

### The correct approach

```
Timeline ────────────────────────────────────────────────────▶
         Jan 2024  ...  Jun 2025    Jul 2025  ...  Dec 2025
         ◄────────── TRAIN ──────────────►◄──── TEST ─────►

Station aggregates computed here ───►  joined onto both splits
```

Aggregate features are computed **only from the training window** and
then joined onto both splits.

In [8]:
trips.agg(
    F.min("start_ts").alias("earliest"),
    F.max("start_ts").alias("latest")
).show(truncate=False)

+-------------------+-------------------+
|earliest           |latest             |
+-------------------+-------------------+
|2024-01-01 00:04:58|2025-12-31 23:39:58|
+-------------------+-------------------+



In [ ]:
# trips = trips.withColumn("random_feature", F.rand(seed=42))

In [9]:
# Adjust CUTOFF after inspecting the date range above.
CUTOFF = "2025-07-01"

train_raw = trips.filter(F.col("start_ts") <  CUTOFF)
test_raw  = trips.filter(F.col("start_ts") >= CUTOFF)

n_train = train_raw.count()
n_test  = test_raw.count()
pct = 100 * n_train / (n_train + n_test)
print(f"Train: {n_train:,}  Test: {n_test:,}  ({pct:.0f}% / {100-pct:.0f}%)")

Train: 656,623  Test: 378,020  (63% / 37%)


In [10]:
# Station aggregates from TRAINING DATA ONLY
station_stats = (
    train_raw
    .groupBy("start_station_id")
    .agg(
        F.count("*").alias("station_trip_count"),
        F.avg("label").alias("station_long_trip_rate")
    )
)

fill_vals = {"station_trip_count": 0.0, "station_long_trip_rate": 0.0}

train_df = train_raw.join(station_stats, on="start_station_id", how="left").fillna(fill_vals)
test_df  = test_raw.join(station_stats,  on="start_station_id", how="left").fillna(fill_vals)

train_df.select("start_station_name", "station_trip_count", "station_long_trip_rate", "label").show(5)

+--------------------+------------------+----------------------+-----+
|  start_station_name|station_trip_count|station_long_trip_rate|label|
+--------------------+------------------+----------------------+-----+
|N Dithridge St & ...|             33390|   0.04165917939502845|  0.0|
|Coltart Ave & For...|             27565|   0.06468347542173046|  0.0|
|Fifth Ave & S Bou...|             30079|   0.06539446125203631|  0.0|
|Zulema St & Colta...|             29350|   0.05652470187393526|  0.0|
|O'Hara St and Uni...|             34711|   0.04335801330990176|  0.0|
+--------------------+------------------+----------------------+-----+
only showing top 5 rows


---
## Part 4 — Spark MLlib Abstractions

| Abstraction | Role | Example |
|---|---|---|
| **Transformer** | DataFrame → mutated DataFrame | `StringIndexer` (post-fit), `VectorAssembler` |
| **Estimator** | DataFrame → fitted Model (itself a Transformer) | `LogisticRegression.fit()` |
| **Pipeline** | Sequential chain of Transformers + Estimators | `Pipeline(stages=[...]).fit(train)` |

**Why Pipeline prevents leakage:**  
A single `pipeline.fit(train_df)` call ensures every `Estimator` inside is fitted
on training data only.  Without it, it is easy to accidentally call `.fit()` on
the full dataset.

### Data-parallel training loop (Spark internals)

```
For each iteration:
  Driver  : broadcast model weights W to all executors
  Executor 1 … N: compute local gradient gi on own data partition  ← parallel
  Driver  : aggregate g1 + g2 + … + gN  →  update W
             ↑ ALL executors wait at this barrier
```

> Synchronisation cost scales linearly with executor count.  
> Beyond a certain cluster size, communication overhead dominates compute.

---
## Part 5 — Pipeline: Logistic Regression Baseline

Why logistic regression as the distributed baseline?
- Gradient computation is **embarrassingly data-parallel**.
- Parameter payload is tiny — one coefficient per feature.
- Convergence is stable, making timing experiments meaningful.

In [12]:
# Cast station ID to string for consistent StringIndexer handling
train_df = train_df.withColumn("start_station_id", F.col("start_station_id").cast("string"))
test_df  = test_df.withColumn("start_station_id",  F.col("start_station_id").cast("string"))

CAT_COLS     = ["start_station_id", "rider_type"]
NUMERIC_COLS = ["month", "day_of_week", "hour",
                "station_trip_count", "station_long_trip_rate"]

indexers = [
    StringIndexer(inputCol=c, outputCol=f"{c}_idx", handleInvalid="keep")
    for c in CAT_COLS
]
encoders = [
    OneHotEncoder(inputCol=f"{c}_idx", outputCol=f"{c}_ohe", handleInvalid="keep")
    for c in CAT_COLS
]
assembler = VectorAssembler(
    inputCols=[f"{c}_ohe" for c in CAT_COLS] + NUMERIC_COLS,
    outputCol="features",
    handleInvalid="keep"
)
lr = LogisticRegression(
    featuresCol="features", labelCol="label",
    maxIter=30, regParam=0.01, elasticNetParam=0.0
)

lr_pipeline = Pipeline(stages=indexers + encoders + [assembler, lr])

print("Pipeline stages:")
for s in lr_pipeline.getStages():
    print(f"  {type(s).__name__}")

Pipeline stages:
  StringIndexer
  StringIndexer
  OneHotEncoder
  OneHotEncoder
  VectorAssembler
  LogisticRegression


In [13]:
t0 = time.time()
lr_model = lr_pipeline.fit(train_df)
lr_train_time = time.time() - t0

print(f"Logistic Regression fit in {lr_train_time:.1f}s")

26/03/31 20:09:04 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
26/03/31 20:09:06 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS


Logistic Regression fit in 4.4s


In [14]:
lr_preds_in_sample = lr_model.transform(train_df)

In [15]:
lr_preds = lr_model.transform(test_df)

lr_preds.select(
    "start_station_name", "rider_type", "hour", "label", "prediction", "probability", "rawPrediction"
).show(20, truncate=False)

+-----------------------+----------+----+-----+----------+----------------------------------------+------------------------------------------+
|start_station_name     |rider_type|hour|label|prediction|probability                             |rawPrediction                             |
+-----------------------+----------+----+-----+----------+----------------------------------------+------------------------------------------+
|Ross St & Fourth Ave   |MEMBER    |23.0|1.0  |0.0       |[0.7299816692010099,0.27001833079899007]|[0.9945295746016709,-0.9945295746016709]  |
|Ross St & Fourth Ave   |MEMBER    |19.0|0.0  |0.0       |[0.7449664892298686,0.25503351077013137]|[1.071944285159295,-1.071944285159295]    |
|Ross St & Fourth Ave   |CASUAL    |18.0|1.0  |1.0       |[0.3813964951465109,0.618603504853489]  |[-0.48362502233683946,0.48362502233683946]|
|Ross St & Fourth Ave   |CASUAL    |18.0|0.0  |1.0       |[0.3813964951465109,0.618603504853489]  |[-0.48362502233683946,0.48362502233683946]|

In [16]:
from pyspark.ml.evaluation import BinaryClassificationEvaluator

auc = BinaryClassificationEvaluator(
    labelCol="label",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC"
).evaluate(lr_preds_in_sample)

print(f"In-sample AUC: {auc:.4f}")

auc = BinaryClassificationEvaluator(
    labelCol="label",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC"
).evaluate(lr_preds)

print(f"AUC: {auc:.4f}")

In-sample AUC: 0.8008
AUC: 0.7851


---
## Part 6 — Which Models Scale Well?

| Algorithm | Distributed Scaling | Core Limitation |
|---|---|---|
| **Logistic Regression** | Excellent — compact gradient payload | Repeated passes; sync overhead |
| **K-Means** | Good — parallel assignment step | Global centroid sync per iteration; init sensitivity |
| **Random Forest** | Good — trees grown in parallel | Split-finding is communication-intensive |
| **Gradient Boosted Trees** | Moderate — sequential boosting | Each tree depends on prior residuals |
| **K-Nearest Neighbours** | Poor — no training, but inference requires global distance computation | Does not parallelise; needs full dataset in memory |

We now fit a **Random Forest** on the same feature pipeline and compare.

In [17]:
rf = RandomForestClassifier(
    featuresCol="features", labelCol="label",
    numTrees=50, maxDepth=8, seed=42
)

rf_pipeline = Pipeline(stages=indexers + encoders + [assembler, rf])

t0 = time.time()
rf_model = rf_pipeline.fit(train_df)
rf_train_time = time.time() - t0

print(f"Random Forest fit in {rf_train_time:.1f}s")

rf_preds = rf_model.transform(test_df)

Random Forest fit in 16.0s


In [18]:
rf_preds_in_sample = rf_model.transform(train_df)
rf_preds = rf_model.transform(test_df)

from pyspark.ml.evaluation import BinaryClassificationEvaluator

auc = BinaryClassificationEvaluator(
    labelCol="label",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC"
).evaluate(rf_preds_in_sample)

print(f"In-sample AUC: {auc:.4f}")

auc = BinaryClassificationEvaluator(
    labelCol="label",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC"
).evaluate(rf_preds)

print(f"AUC: {auc:.4f}")

In-sample AUC: 0.7997


AUC: 0.7799


In [19]:
# Total cores available to this Spark session
print("Master:         ", spark.sparkContext.master)
print("Default parallelism:", spark.sparkContext.defaultParallelism)
print("Executor cores: ", spark.sparkContext.getConf().get("spark.executor.cores", "not set (local mode)"))

Master:          local[*]
Default parallelism: 18
Executor cores:  not set (local mode)


In [20]:
# pip install xgboost  (if not already installed)
from xgboost.spark import SparkXGBClassifier

xgb = SparkXGBClassifier(
    features_col="features",
    label_col="label",
    num_workers=18,        # number of Spark executors to use
    n_estimators=200,
    max_depth=5,
    learning_rate=0.1,
    subsample=0.8,
    use_gpu=False
)

xgb_pipeline = Pipeline(stages=indexers + encoders + [assembler, xgb])

t0 = time.time()
xgb_model = xgb_pipeline.fit(train_df)
xgb_train_time = time.time() - t0

xgb_preds_in_sample = xgb_model.transform(train_df)
xgb_preds = xgb_model.transform(test_df)

auc = BinaryClassificationEvaluator(
    labelCol="label",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC"
).evaluate(xgb_preds_in_sample)

print(f"In-sample AUC: {auc:.4f}")

auc = BinaryClassificationEvaluator(
    labelCol="label",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC"
).evaluate(xgb_preds)

print(f"AUC: {auc:.4f}")

2026-03-31 20:17:45,869 INFO XGBoost-PySpark: _fit Running xgboost-3.2.0 on 18 workers with
	booster params: {'objective': 'binary:logistic', 'device': 'cpu', 'learning_rate': 0.1, 'max_depth': 5, 'subsample': 0.8, 'use_gpu': False, 'nthread': 1}
	train_call_kwargs_params: {'verbose_eval': True, 'num_boost_round': 200}
	dmatrix_kwargs: {'nthread': 1, 'missing': nan}
2026-03-31 20:17:50,070 INFO XGBoost-PySpark: _train_booster Training on CPUs18]
[20:17:51] Task 17 got rank 9
[20:17:51] Task 16 got rank 8
[20:17:51] Task 2 got rank 10
[20:17:51] Task 15 got rank 7
[20:17:51] Task 3 got rank 11
[20:17:51] Task 14 got rank 6
[20:17:51] Task 4 got rank 12
[20:17:51] Task 13 got rank 5
[20:17:51] Task 12 got rank 4
[20:17:51] Task 5 got rank 13
[20:17:51] Task 11 got rank 3
[20:17:51] Task 6 got rank 14
[20:17:51] Task 10 got rank 2[20:17:51] Task 1 got rank 1

[20:17:51] Task 7 got rank 15
[20:17:51] Task 8 got rank 16
[20:17:51] Task 0 got rank 0
[20:17:51] Task 9 got rank 17
/opt/anacond

In-sample AUC: 0.8264


2026-03-31 20:18:00,624 INFO XGBoost-PySpark: predict_udf Do the inference on the CPUs


AUC: 0.8039


---
## Part 7 — Evaluation at Scale

Two distributed nuances:

1. **K-fold cross-validation** requires K full model fits over K full dataset scans.  
   At scale this is often too expensive — prefer a **single out-of-time holdout**.

2. **Class imbalance** makes naive accuracy misleading.  
   Always report AUC alongside accuracy.

In [21]:
auc_eval = BinaryClassificationEvaluator(
    labelCol="label", rawPredictionCol="rawPrediction", metricName="areaUnderROC"
)
acc_eval = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="accuracy"
)

lr_auc = auc_eval.evaluate(lr_preds)
lr_acc = acc_eval.evaluate(lr_preds)
rf_auc = auc_eval.evaluate(rf_preds)
rf_acc = acc_eval.evaluate(rf_preds)

print(f"{'Model':<25} {'AUC':>8} {'Accuracy':>10} {'Train (s)':>12}")
print("-" * 60)
print(f"{'Logistic Regression':<25} {lr_auc:>8.4f} {lr_acc:>10.4f} {lr_train_time:>12.1f}")
print(f"{'Random Forest':<25} {rf_auc:>8.4f} {rf_acc:>10.4f} {rf_train_time:>12.1f}")

Model                          AUC   Accuracy    Train (s)
------------------------------------------------------------
Logistic Regression         0.7851     0.8663          4.4
Random Forest               0.7799     0.8669         16.0


In [22]:
print("LR confusion matrix:")
lr_preds.groupBy("label", "prediction").count().orderBy("label", "prediction").show()

print("RF confusion matrix:")
rf_preds.groupBy("label", "prediction").count().orderBy("label", "prediction").show()

LR confusion matrix:
+-----+----------+------+
|label|prediction| count|
+-----+----------+------+
|  0.0|       0.0|316161|
|  0.0|       1.0|  8073|
|  1.0|       0.0| 42476|
|  1.0|       1.0| 11310|
+-----+----------+------+

RF confusion matrix:


+-----+----------+------+
|label|prediction| count|
+-----+----------+------+
|  0.0|       0.0|316280|
|  0.0|       1.0|  7954|
|  1.0|       0.0| 42345|
|  1.0|       1.0| 11441|
+-----+----------+------+



In [23]:
# Feature importances from Random Forest
feature_names = [f"{c}_ohe" for c in CAT_COLS] + NUMERIC_COLS
importances = sorted(
    zip(feature_names, rf_model.stages[-1].featureImportances.toArray()),
    key=lambda x: x[1], reverse=True
)

print(f"{'Feature':<30} {'Importance':>12}")
print("-" * 45)
for name, imp in importances:
    print(f"{name:<30} {imp:>12.4f}")

Feature                          Importance
---------------------------------------------
start_station_id_ohe                 0.0084
day_of_week                          0.0069
month                                0.0018
rider_type_ohe                       0.0008
station_trip_count                   0.0006
station_long_trip_rate               0.0003
hour                                 0.0001


---
## Distributed Optimization Trade-offs: Synchronous vs Asynchronous

```
SYNCHRONOUS (Spark MLlib default)
Worker 1: ████████░░░░  (fast)
Worker 2: ██████░░░░░░  (faster)
Worker 3: ████████████  (STRAGGLER)
                      ↑ all workers wait here

ASYNCHRONOUS (parameter server)
Worker 1: ████→push→████→push→…
Worker 2: ████→push→████→push→…
Worker 3: ██████████→push→…       (no waiting)
```

| | Synchronous | Asynchronous |
|---|---|---|
| Convergence stability | High | Lower — stale gradients |
| Hardware utilisation | Poor with stragglers | Maximised |
| Reproducibility | Yes | No |
| Used by | Spark MLlib, Horovod AllReduce | TF ParameterServer, Hogwild! |

> A **stale gradient** is computed from outdated model weights.  
> It can point in the wrong direction and slow or destabilise convergence.

---
## Part 8 — Clustering: Station Profiles

We cluster POGOH stations by their operational profile using K-Means.

K-Means distributed loop:
```
Init: broadcast K centroids
Each iteration:
  1. Executors assign rows to nearest centroid  (fully parallel)
  2. Executors aggregate local sums + counts
  3. Driver computes new global centroids + broadcasts
```

Bottleneck: **global centroid sync** per iteration.  
With K clusters and D features, each broadcast carries K×D floats.
This is cheap for small K/D but non-trivial for high-dimensional embeddings.

In [24]:
# Build per-station profiles from training data only
station_profiles = (
    train_df
    .groupBy("start_station_id", "start_station_name")
    .agg(
        F.count("*").alias("n_trips"),
        F.avg("label").alias("long_trip_rate"),
        F.avg("hour").alias("avg_hour"),
        F.stddev("hour").alias("stddev_hour"),
        F.avg("day_of_week").alias("avg_day_of_week"),
        F.avg(
            F.when(F.col("rider_type") == "CASUAL", 1.0).otherwise(0.0)
        ).alias("casual_rate")
    )
    .filter(F.col("n_trips") >= 10)
    .fillna(0.0)
)

print(f"Stations in profile: {station_profiles.count()}")
station_profiles.show(10, truncate=False)

Stations in profile: 64
+----------------+----------------------------------------+-------+--------------------+------------------+------------------+------------------+--------------------+
|start_station_id|start_station_name                      |n_trips|long_trip_rate      |avg_hour          |stddev_hour       |avg_day_of_week   |casual_rate         |
+----------------+----------------------------------------+-------+--------------------+------------------+------------------+------------------+--------------------+
|40              |7th St & Penn Ave                       |8390   |0.3172824791418355  |14.79439809296782 |4.994529570443616 |4.080214541120381 |0.22753277711561382 |
|1               |Pierce St & Summerlea St                |5007   |0.11364090273616936 |12.639304973037747|5.7186775594211365|4.116836428999401 |0.0683043738765728  |
|44              |Hamilton Ave & Fifth Ave                |1147   |0.3722755013077594  |13.710549258936355|4.144204973676548 |3.9886660854402

In [25]:
CLUSTER_FEATURES = [
    "n_trips", "long_trip_rate", "avg_hour",
    "stddev_hour", "avg_day_of_week", "casual_rate"
]

cluster_assembler = VectorAssembler(
    inputCols=CLUSTER_FEATURES, outputCol="raw_cluster_features"
)

# StandardScaler is important here.
# n_trips is on a very different scale from rate/proportion features.
# Without scaling, K-Means clusters on trip volume and ignores everything else.
scaler = StandardScaler(
    inputCol="raw_cluster_features", outputCol="cluster_features",
    withStd=True, withMean=True
)

kmeans = KMeans(
    featuresCol="cluster_features", predictionCol="cluster",
    k=4, maxIter=30, seed=42
)

cluster_pipeline = Pipeline(stages=[cluster_assembler, scaler, kmeans])
cluster_model    = cluster_pipeline.fit(station_profiles)
station_clusters = cluster_model.transform(station_profiles)

station_clusters.select(
    "start_station_name", "n_trips", "long_trip_rate", "avg_hour", "casual_rate", "cluster"
).orderBy("cluster", "start_station_name").show(40, truncate=False)

+----------------------------------------+-------+--------------------+------------------+--------------------+-------+
|start_station_name                      |n_trips|long_trip_rate      |avg_hour          |casual_rate         |cluster|
+----------------------------------------+-------+--------------------+------------------+--------------------+-------+
|10th St & Penn Ave                      |8740   |0.31430205949656753 |14.75938215102975 |0.28043478260869564 |0      |
|17th St & Penn Ave                      |6293   |0.371523915461624   |15.245669791832194|0.3151120292388368  |0      |
|21st St & Penn Ave                      |8365   |0.32946802151823074 |14.393424985056784|0.29850567842199643 |0      |
|7th St & Penn Ave                       |8390   |0.3172824791418355  |14.79439809296782 |0.22753277711561382 |0      |
|Forbes Ave & Market Square              |9259   |0.4467005076142132  |14.986931634085755|0.2843719624149476  |0      |
|Glasshouse                             

In [26]:
cluster_eval = ClusteringEvaluator(
    featuresCol="cluster_features", predictionCol="cluster", metricName="silhouette"
)

silhouette = cluster_eval.evaluate(station_clusters)
print(f"Silhouette score (k=4): {silhouette:.4f}")
# +1 = well-separated,  0 = overlapping,  <0 = likely misclustered

Silhouette score (k=4): 0.3739


In [ ]:
# Summarise cluster characteristics — can you label each group?
station_clusters.groupBy("cluster").agg(
    F.count("*").alias("n_stations"),
    F.round(F.avg("n_trips"),          0).alias("avg_trips"),
    F.round(F.avg("long_trip_rate"),   3).alias("long_trip_rate"),
    F.round(F.avg("avg_hour"),         1).alias("avg_dep_hour"),
    F.round(F.avg("casual_rate"),      3).alias("casual_rate")
).orderBy("cluster").show()

+-------+----------+---------+--------------+------------+-----------+
|cluster|n_stations|avg_trips|long_trip_rate|avg_dep_hour|casual_rate|
+-------+----------+---------+--------------+------------+-----------+
|      0|        13|   6985.0|         0.423|        14.9|      0.328|
|      1|        18|   5270.0|           0.3|        14.6|      0.108|
|      2|        11|  32865.0|         0.058|        14.2|      0.014|
|      3|        22|   4975.0|         0.289|        13.5|      0.148|
+-------+----------+---------+--------------+------------+-----------+



26/04/04 08:36:48 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 698753 ms exceeds timeout 120000 ms
26/04/04 08:36:48 WARN SparkContext: Killing executors is not supported by current scheduler.
26/04/04 08:36:48 ERROR Inbox: Ignoring error
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:53)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:359)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRefByURI(RpcEnv.scala:102)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRef(RpcEnv.scala:110)
	at org.apache.spark.util.RpcUtils$.makeDriverRef(RpcUtils.scala:36)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.driverEndpoint$lzycompute(BlockManagerMasterEndpoint.scala:132)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.org$apache$spark$storage$BlockManagerMasterEndpoint$$

---
## Part 9 — End-to-End Operational Workflow

The pipeline we built follows the four-step POGOH workflow from the lecture:

```
1. INGEST & CLEAN             2. FEATURE PIPELINE
   read CSV                      temporal features
   rename columns                station aggregates
   parse timestamps              categorical encoding
   filter bad rows                     │
        │                              │
        └────────── TIME-BASED SPLIT ──┘
                   (train < CUTOFF)
                        │
        3. ASSEMBLE & TRAIN       4. EVALUATE
           Pipeline.fit()            AUC + accuracy
           LR vs RF                  Confusion matrix
           Data parallelism          Feature importances
```

### Batch vs incremental retraining

| | Periodic batch | Incremental / online |
|---|---|---|
| Latency to adapt | Hours–days | Minutes–seconds |
| MLlib support | Native | Not native |
| Auditability | High | Low |
| **Right for POGOH?** | **Yes** | No — seasonal, not intra-day drift |

For POGOH, **periodic batch retraining** is the correct operational choice.

In [ ]:
# Save the fitted pipeline to disk for later scoring
# lr_model.write().overwrite().save("models/pogoh_lr_pipeline")

# Reload without refitting:
# from pyspark.ml import PipelineModel
# loaded = PipelineModel.load("models/pogoh_lr_pipeline")
# new_preds = loaded.transform(new_trips_df)

print("Coefficient vector size:", lr_model.stages[-1].coefficients.size)
print("(One coefficient per encoded feature — small enough to broadcast cheaply)")

Coefficient vector size: 70
(One coefficient per encoded feature — small enough to broadcast cheaply)


26/03/31 18:17:49 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 240732 ms exceeds timeout 120000 ms
26/03/31 18:17:49 WARN SparkContext: Killing executors is not supported by current scheduler.
26/03/31 18:17:58 ERROR Inbox: Ignoring error
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:53)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:359)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRefByURI(RpcEnv.scala:102)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRef(RpcEnv.scala:110)
	at org.apache.spark.util.RpcUtils$.makeDriverRef(RpcUtils.scala:36)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.driverEndpoint$lzycompute(BlockManagerMasterEndpoint.scala:132)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.org$apache$spark$storage$BlockManagerMasterEndpoint$$

---
## Systems Discussion Questions

### Constraints & skew
1. Which of the three constraints (memory, compute, network) would bite first
   when scaling this pipeline to 10 years of full POGOH trip data?  Justify your answer.
2. The busiest station has far more trips than the quietest.  How does this skew affect
   the `groupBy` for `station_stats`?  Name two mitigation strategies.

### Leakage
3. A colleague joins `End Station Name` into the feature set.  Why is this leakage even
   if the column appears in both train and test DataFrames?
4. You want a rolling 7-day trip count as a feature.  Describe the procedure that
   guarantees no temporal leakage.

### Distributed optimisation
5. One executor is 5× slower than its peers.  How does this affect synchronous training?
   How would asynchronous training behave differently?
6. For 10 K rows, does distributed training offer any speedup over single-machine sklearn?
   At what data volume would your answer change?

### Model selection
7. Looking at the comparison table, does RF meaningfully outperform LR here?  At what
   scale or feature complexity would you expect the gap to widen?
8. A stakeholder proposes KNN.  Write a paragraph explaining why KNN is unsuitable at scale.

### Evaluation & operations
9. We used a single holdout rather than K-fold CV.  What is the statistical cost?
   What is the systems justification for accepting it?
10. How often should this model be retrained?  What signals would prompt earlier retraining?
    (What constitutes concept drift in the bike-share domain?)

---
## Optional Exercises

### Core
1. Build the full LR pipeline and report AUC, accuracy, and the confusion matrix.
2. Explain in one paragraph why the time-based split + training-only aggregates
   combination prevents temporal leakage.
3. Name two columns in the raw POGOH data that constitute leakage.  Explain each.

### Intermediate
4. Add an `is_weekend` feature (1 = Saturday or Sunday).  Rebuild the pipeline
   and compare AUC before and after.
5. Experiment with `spark.sql.shuffle.partitions` at 10, 50, and 200.
   Does training time change noticeably for this dataset size?  Why or why not?
6. Call `train_df.cache()` before `pipeline.fit()`.  Does it reduce fitting time?
   When would caching have a larger impact?

### Advanced
7. Implement a manual 3-fold time-series cross-validator using three equal-width
   time windows.  Report mean ± stddev of AUC across folds.
8. Replace `LogisticRegression` with `GBTClassifier`.
   Compare AUC, accuracy, and wall-clock training time.  What does this tell you about
   the compute/accuracy trade-off at this data scale?
9. Sweep K-Means over k = 3, 4, 5, 6, 7.  Plot silhouette vs k and identify the elbow.
   Characterise each cluster at your chosen k.
10. Load `pogoh_full.csv` instead of the sample.  How do training time and AUC change?
    Does more historical data improve the station aggregate features?

---
## Key Takeaways

**ML is a systems problem:** At scale, managing data movement, memory, and communication
matters just as much as optimising algorithm loss curves.

**Feature engineering dominates:** Grouped aggregations trigger shuffles, hot-key stations
create data skew, and iterative recomputation of features increases memory pressure.

**Parallelism dictates scalability:** Data parallelism is the standard Spark tool, but
synchronisation overhead and stragglers limit how far it scales.

**Design governs success:** Correct leakage prevention, the right train/test strategy,
and matching algorithm to communication budget all have larger production impact than
hyperparameter tuning.

---
## References

- *Learning Spark* — Chapter 10 (MLlib and distributed pipelines)  
- *Mining of Massive Datasets* — Chapters 7 (clustering) and 12 (large-scale ML)  
- POGOH Pittsburgh bike-share open data: https://pogoh.com/open-data/  
- Spark MLlib documentation: https://spark.apache.org/docs/latest/ml-guide.html